In [28]:
import re

from dataclasses import dataclass

import time
from statistics import mean, median

from llama_index.core import (
    Document,
    SimpleDirectoryReader,
    TreeIndex,
    VectorStoreIndex,
    SummaryIndex,
)
from llama_index.core.node_parser import NodeParser, SentenceSplitter
from llama_index.core.schema import TextNode, NodeWithScore
from llama_index.core.postprocessor.types import BaseNodePostprocessor
from llama_index.llms.google_genai import GoogleGenAI

### Task 1

In [2]:
class MarkdownHeadingParser(NodeParser):
    def _parse_nodes(self, nodes, show_progress=False, **kwargs):
        result = []

        for node in nodes:
            current_headers = {}
            sections = re.split(r"(?m)(?=^#{1,6}\s+)", node.get_content())

            for section in sections:
                section = section.strip()

                if not section:
                    continue

                match = re.match(r"^(#{1,6})\s+(.+)", section)

                if match:
                    level = len(match.group(1))
                    title = match.group(2).strip()

                    current_headers[level] = title

                    # Remove deeper headings
                    for key in list(current_headers):
                        if key > level:
                            del current_headers[key]

                    context = "\n".join(
                        f"{'#' * k} {v}" for k, v in sorted(current_headers.items())
                    )

                    section = f"{context}\n\n{section}"

                result.append(TextNode(text=section, metadata=node.metadata.copy()))

        return result

In [3]:
document = Document(
    text="""
# Python

Python is a programming language.

## Variables

Variables store values.

## Functions

Functions allow code reuse.

# Machine Learning

Machine learning learns patterns from data.

## Supervised Learning

The model learns from labeled data.

### Algorithms

1. LinearRegression
2. Ridge and Lasso

"""
)

parser = MarkdownHeadingParser()

nodes = parser.get_nodes_from_documents([document])

for node in nodes:
    print("=" * 40)
    print(node.get_content())

# Python

# Python

Python is a programming language.
# Python
## Variables

## Variables

Variables store values.
# Python
## Functions

## Functions

Functions allow code reuse.
# Machine Learning

# Machine Learning

Machine learning learns patterns from data.
# Machine Learning
## Supervised Learning

## Supervised Learning

The model learns from labeled data.
# Machine Learning
## Supervised Learning
### Algorithms

### Algorithms

1. LinearRegression
2. Ridge and Lasso


### Task 2

In [5]:
llm = GoogleGenAI(model="gemini-3.5-flash-lite", temperature=0)

In [10]:
async def route_query(query: str):
    prompt = f"""
    Classify the following query into exactly one category:

    - vector: specific information or factual lookup
    - summary: wants an overall summary or overview

    Query:
    {query}

    Answer only: vector or summary
    """

    response = await llm.acomplete(prompt)

    route = response.text.strip().lower()

    if route == "summary":
        return "summary"

    return "vector"

In [11]:
query = "Give me an overview of the entire document."

route = await route_query(query)

if route == "summary":
    # atual route to engine
    response = "Routed for summary"
else:
    # atual route to engine
    response = "Routed for vector search"

print(response)

Routed for summary


### Task 3

In [12]:
def get_retrieved_node_ids(response) -> set[str]:
    """
    Return the IDs of nodes actually retrieved by LlamaIndex.
    """

    return {source_node.node.node_id for source_node in response.source_nodes}


def extract_citations(answer: str) -> list[str]:
    """
    Extract citations from the generated answer.
    """

    pattern = r"\[citation:\s*([^\]]+)\]"

    return [citation.strip() for citation in re.findall(pattern, answer)]


def validate_citations(response) -> dict:
    """
    Validate generated citations against the nodes
    actually retrieved by the query engine.
    """

    answer = response.response

    retrieved_ids = get_retrieved_node_ids(response)

    citations = extract_citations(answer)

    valid_citations = [citation for citation in citations if citation in retrieved_ids]

    invalid_citations = [citation for citation in citations if citation not in retrieved_ids]

    return {
        "is_valid": len(invalid_citations) == 0,
        "total_citations": len(citations),
        "valid_citations": valid_citations,
        "invalid_citations": invalid_citations,
        "retrieved_nodes": list(retrieved_ids),
    }

### Task 4

In [19]:
@dataclass
class DocumentMetadata:
    source: str
    date: str
    document_type: str
    tenant_id: str


class IngestionPipeline:
    def __init__(
        self,
        chunk_size: int = 512,
        chunk_overlap: int = 50,
    ):
        self.node_parser = SentenceSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

    def _load_document(
        self,
        file_path: str,
        metadata: DocumentMetadata,
    ) -> Document:
        documents = SimpleDirectoryReader(input_dir=file_path).load_data()

        for document in documents:
            document.metadata.update(
                {
                    "document_type": metadata.document_type,
                    "document_date": metadata.date,
                    "tenant_id": metadata.tenant_id,
                }
            )

        return documents

    def _create_nodes(
        self,
        document: list[Document],
    ):
        nodes = self.node_parser.get_nodes_from_documents(document)

        return nodes

    def ingest(
        self,
        file_path: str,
        metadata: DocumentMetadata,
    ):
        document = self._load_document(
            file_path=file_path,
            metadata=metadata,
        )

        nodes = self._create_nodes(document)

        return nodes

### Task 5

In [21]:
def benchmark_response_modes(
    query_engine,
    queries: list[str],
    runs: int = 3,
) -> dict:
    """
    Benchmark query latency for LlamaIndex response modes:
    - compact
    - tree_summarize

    The same query set is executed for both modes.
    """

    results = {
        "compact": [],
        "tree_summarize": [],
    }

    for response_mode in results:
        engine = query_engine.as_query_engine(response_mode=response_mode)

        # Warm-up run
        engine.query(queries[0])

        for query in queries:
            for _ in range(runs):
                start = time.perf_counter()

                response = engine.query(query)

                latency = time.perf_counter() - start

                results[response_mode].append(
                    {
                        "query": query,
                        "latency_seconds": latency,
                        "response": response,
                    }
                )

    benchmark = {}

    for mode, measurements in results.items():
        latencies = [item["latency_seconds"] for item in measurements]

        benchmark[mode] = {
            "total_runs": len(latencies),
            "mean_latency_seconds": mean(latencies),
            "median_latency_seconds": median(latencies),
            "min_latency_seconds": min(latencies),
            "max_latency_seconds": max(latencies),
            "all_latencies_seconds": latencies,
        }

    return benchmark

### Task 7

In [23]:
def query_hierarchical_document(
    query: str,
    tree_index: TreeIndex,
    similarity_top_k: int = 5,
):
    """
    Query the hierarchical document through TreeIndex.
    """

    if tree_index is None:
        raise RuntimeError("TreeIndex has not been initialized.")

    query_engine = tree_index.as_query_engine(
        response_mode="tree_summarize",
        similarity_top_k=similarity_top_k,
    )

    response = query_engine.query(query)

    source_nodes = []

    for source_node in response.source_nodes:
        node = source_node.node

        source_nodes.append(
            {
                "node_id": node.node_id,
                "score": source_node.score,
                "text": node.get_content(),
                "metadata": node.metadata,
            }
        )

    return {
        "query": query,
        "answer": str(response),
        "source_nodes": source_nodes,
    }


# FasrAPI endpoint


async def query_document(query: str, similarity_top_k: int = 5):
    result = query_hierarchical_document(
        query=query,
        similarity_top_k=similarity_top_k,
    )

    return result

### Task 8

In [25]:
def benchmark_indexing_cost(
    documents,
    vector_index_kwargs: dict | None = None,
    summary_index_kwargs: dict | None = None,
    runs: int = 3,
) -> dict:
    vector_index_kwargs = vector_index_kwargs or {}
    summary_index_kwargs = summary_index_kwargs or {}

    results = {
        "VectorStoreIndex": [],
        "SummaryIndex": [],
    }

    def get_token_usage(callback_manager):
        total_input_tokens = 0
        total_output_tokens = 0

        for handler in getattr(
            callback_manager,
            "handlers",
            [],
        ):
            if hasattr(handler, "total_tokens"):
                total_input_tokens += getattr(
                    handler,
                    "prompt_tokens",
                    0,
                )

                total_output_tokens += getattr(
                    handler,
                    "completion_tokens",
                    0,
                )

        return {
            "input_tokens": total_input_tokens,
            "output_tokens": total_output_tokens,
            "total_tokens": (total_input_tokens + total_output_tokens),
        }

    for _ in range(runs):
        start = time.perf_counter()

        vector_index = VectorStoreIndex.from_documents(
            documents,
            **vector_index_kwargs,
        )

        elapsed = time.perf_counter() - start

        token_usage = get_token_usage(vector_index._callback_manager)

        results["VectorStoreIndex"].append(
            {
                "time_seconds": elapsed,
                **token_usage,
            }
        )

    for _ in range(runs):
        start = time.perf_counter()

        summary_index = SummaryIndex.from_documents(
            documents,
            **summary_index_kwargs,
        )

        elapsed = time.perf_counter() - start

        token_usage = get_token_usage(summary_index._callback_manager)

        results["SummaryIndex"].append(
            {
                "time_seconds": elapsed,
                **token_usage,
            }
        )

    # ---------------------------------------------------------
    # Aggregate results
    # ---------------------------------------------------------

    benchmark = {}

    for index_type, measurements in results.items():
        times = [item["time_seconds"] for item in measurements]

        input_tokens = [item["input_tokens"] for item in measurements]

        output_tokens = [item["output_tokens"] for item in measurements]

        total_tokens = [item["total_tokens"] for item in measurements]

        benchmark[index_type] = {
            "runs": runs,
            "mean_time_seconds": mean(times),
            "min_time_seconds": min(times),
            "max_time_seconds": max(times),
            "mean_input_tokens": mean(input_tokens),
            "mean_output_tokens": mean(output_tokens),
            "mean_total_tokens": mean(total_tokens),
            "raw_runs": measurements,
        }

    return benchmark

### Task 9

In [29]:
class MinimumScorePostprocessor(BaseNodePostprocessor):
    """
    Filters out retrieved nodes whose relevance score is
    below the configured minimum threshold.
    """

    min_score: float

    def _postprocess_nodes(
        self,
        nodes: list[NodeWithScore],
        query_bundle=None,
    ) -> list[NodeWithScore]:
        filtered_nodes = [
            node for node in nodes if node.score is not None and node.score >= self.min_score
        ]

        return filtered_nodes